Soil Category

In [22]:
from pathlib import Path
import rasterio
import numpy as np
import pandas as pd

# 路径
soil_path = Path("/Users/wangze/Dropbox/Emi/LandControl/geodata/soil_jiangsu/JS.tif")

# 打开栅格
with rasterio.open(soil_path) as src:
    arr = src.read(1)     # 读取 band 1
    nodata = src.nodata   # 读取 nodata 值

# 去掉 nodata
if nodata is not None:
    arr = arr[arr != nodata]

# 获取唯一值（即土壤分类编码）
values, counts = np.unique(arr, return_counts=True)

df = pd.DataFrame({
    "value": values,
    "count": counts
})

print(df.head())
print("Total unique soil classes:", len(df))

# 保存以备后用
out_csv = soil_path.parent / "JS_value_counts.csv"
df.to_csv(out_csv, index=False)
print("Saved to:", out_csv)


   value    count
0      1   215336
1      2   780703
2     14   625053
3     17   399086
4     18  8147800
Total unique soil classes: 55
Saved to: /Users/wangze/Dropbox/Emi/LandControl/geodata/soil_jiangsu/JS_value_counts.csv


In [23]:
from pathlib import Path
import pandas as pd

# 路径
base_dir = Path("/Users/wangze/Dropbox/Emi/LandControl")

soil_xlsx = base_dir / "excel_data/jiangsu_land_index/Land_Index.xlsx"
value_counts_csv = base_dir / "geodata/soil_jiangsu/JS_value_counts.csv"

# 输出路径：新文件
output_xlsx = base_dir / "geodata/soil_jiangsu/JS_value.xlsx"

# -----------------------------
# 1) 读取你手打的 soil sheet
# -----------------------------
soil_df = pd.read_excel(soil_xlsx, sheet_name="soil")

# -----------------------------
# 2) 读取 value_counts（value 和 count）
# -----------------------------
vc_df = pd.read_csv(value_counts_csv)

# -----------------------------
# 3) Merge
# -----------------------------
soil_full = soil_df.merge(vc_df, on="value", how="left")

print(soil_full.head())
print("N rows:", len(soil_full))

# -----------------------------
# 4) 保存为新的 Excel
# -----------------------------
soil_full.to_excel(output_xlsx, index=False)
print("Saved merged table to:", output_xlsx)


   value     土纲     土类     亚类  Score    count
0      1    淋溶土     棕壤   棕壤性土      4   215336
1      2    淋溶土     棕壤   棕壤性土      4   780703
2     14    淋溶土    暗棕壤    暗棕壤      4   625053
3     17    水成土    沼泽土    沼泽土      1   399086
4     18  湖泊、水库  湖泊、水库  湖泊、水库      1  8147800
N rows: 55
Saved merged table to: /Users/wangze/Dropbox/Emi/LandControl/geodata/soil_jiangsu/JS_value.xlsx


In [24]:
from pathlib import Path
import pandas as pd
import numpy as np
import rasterio

base_dir = Path("/Users/wangze/Dropbox/Emi/LandControl")

# 1. 路径
soil_raster = base_dir / "geodata/soil_jiangsu/JS.tif"
value_xlsx  = base_dir / "geodata/soil_jiangsu/JS_value.xlsx"

# 2. 读取 value–Score 映射表
val_df = pd.read_excel(value_xlsx)   # 应该包含列: value, Score, ...
print(val_df.head())

# 建立字典 {value: score}
value_to_score = dict(zip(val_df["value"], val_df["Score"]))

# 3. 读取原始土壤栅格
with rasterio.open(soil_raster) as src:
    soil = src.read(1)        # 分类值
    profile = src.profile
    nodata_in = src.nodata

print("Original nodata:", nodata_in)

# 4. 初始化适宜性栅格
suit = np.full(soil.shape, np.nan, dtype="float32")

# 5. 按 value→Score 重分类
for v, s in value_to_score.items():
    suit[soil == v] = float(s)

# 把输入栅格的 nodata 也设为 NaN
if nodata_in is not None:
    suit[soil == nodata_in] = np.nan

# 6. 输出时的 nodata 值
nodata_out = -9999.0
suit_out = suit.copy()
suit_out[np.isnan(suit_out)] = nodata_out

# 7. 写出重分类后的栅格
suit_path = soil_raster.parent / "JS_soil_suitability.tif"

profile_out = profile.copy()
profile_out.update(dtype="float32", nodata=nodata_out)

with rasterio.open(suit_path, "w", **profile_out) as dst:
    dst.write(suit_out, 1)

print("Saved soil suitability raster to:", suit_path)

# 可选：看一下适宜性取值范围，确认正常
print("Unique scores in raster:", np.unique(suit_out[suit_out != nodata_out]))


   value     土纲     土类     亚类  Score    count
0      1    淋溶土     棕壤   棕壤性土      4   215336
1      2    淋溶土     棕壤   棕壤性土      4   780703
2     14    淋溶土    暗棕壤    暗棕壤      4   625053
3     17    水成土    沼泽土    沼泽土      1   399086
4     18  湖泊、水库  湖泊、水库  湖泊、水库      1  8147800
Original nodata: 255.0
Saved soil suitability raster to: /Users/wangze/Dropbox/Emi/LandControl/geodata/soil_jiangsu/JS_soil_suitability.tif
Unique scores in raster: [1. 2. 3. 4. 5.]


In [26]:
import geopandas as gpd
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterstats import zonal_stats
from pathlib import Path

base = Path("/Users/wangze/Dropbox/Emi/LandControl/")

# 路径
grid_path = base / "geodata/Jiangsu_CGCS2000/Jiangsu_grid_500m_ID.gpkg"
suit_path_orig = base / "geodata/soil_jiangsu/JS_soil_suitability.tif"
xlsx_out = base / "geodata/soil_jiangsu/Jiangsu_grid_500m_with_soil.xlsx"

# 1. 读取 grid，取 CRS
grid = gpd.read_file(grid_path)
grid_crs = grid.crs
print("Grid CRS:", grid_crs)

# 2. 检查并必要时重投影 soil suitability raster
with rasterio.open(suit_path_orig) as src:
    ras_crs = src.crs
    ras_meta = src.meta.copy()
    nodata = src.nodata
    print("Raster CRS:", ras_crs, "  nodata:", nodata)

    if ras_crs != grid_crs:
        print("CRS differ → reproject raster to grid CRS")

        # 输出重投影文件
        reproj_path = suit_path_orig.with_name("JS_soil_suitability_gridCRS.tif")

        transform, width, height = calculate_default_transform(
            ras_crs, grid_crs, src.width, src.height, *src.bounds
        )

        ras_meta.update({
            "crs": grid_crs,
            "transform": transform,
            "width": width,
            "height": height,
            "nodata": nodata
        })

        with rasterio.open(reproj_path, "w", **ras_meta) as dst:
            for i in range(1, src.count + 1):
                reproject(
                    source=rasterio.band(src, i),
                    destination=rasterio.band(dst, i),
                    src_transform=src.transform,
                    src_crs=ras_crs,
                    dst_transform=transform,
                    dst_crs=grid_crs,
                    resampling=Resampling.nearest,   # 适用于分类/评分数据
                    src_nodata=nodata,
                    dst_nodata=nodata,
                )
        suit_path = reproj_path
    else:
        print("CRS already match, no reprojection needed.")
        suit_path = suit_path_orig

print("Raster used for zonal stats:", suit_path)

# 3. zonal stats
zs = zonal_stats(
    vectors=grid,
    raster=str(suit_path),
    stats=["mean", "std"],
    nodata=nodata,      # 用栅格自身的 nodata
    geojson_out=False
)

grid["soil_suit_mean"] = [z["mean"] for z in zs]
grid["soil_suit_sd"]   = [z["std"]  for z in zs]

# 4. 导出为 xlsx（去掉 geometry）
df = grid.drop(columns="geometry")
df.to_excel(xlsx_out, index=False)

print("Saved Excel:", xlsx_out)
print(df.head())


Grid CRS: EPSG:4547
Raster CRS: PROJCS["Krasovsky_1940_Albers",GEOGCS["Unknown datum based upon the Krassowsky 1940 ellipsoid",DATUM["Not_specified_based_on_Krassowsky_1940_ellipsoid",SPHEROID["Krassowsky 1940",6378245,298.3,AUTHORITY["EPSG","7024"]],AUTHORITY["EPSG","6024"]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4024"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",0],PARAMETER["longitude_of_center",105],PARAMETER["standard_parallel_1",25],PARAMETER["standard_parallel_2",47],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]   nodata: -9999.0
CRS differ → reproject raster to grid CRS
Raster used for zonal stats: /Users/wangze/Dropbox/Emi/LandControl/geodata/soil_jiangsu/JS_soil_suitability_gridCRS.tif
Saved Excel: /Users/wangze/Dropbox/Emi/LandControl/geodata/soil_jiangsu/Jiangsu_grid_500m_with_soil.xlsx
   gr